In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as psx
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from imblearn.over_sampling import SMOTE

In [2]:
data = pd.read_csv("Data/Validation.csv")
data.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,3217900.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,91940.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,505670.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,3217900.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,3217900.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [3]:
data.describe().astype(int)

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,44993,44993,44993,44993,44993,44993,44993,44993,44993
mean,27,79908,5,881077,11,0,5,632,0
std,5,63322,5,580582,2,0,3,50,0
min,20,8000,0,45970,5,0,2,390,0
25%,24,47195,1,459700,8,0,3,601,0
50%,26,67046,4,735520,11,0,4,640,0
75%,30,95778,8,1125069,12,0,8,670,0
max,94,2448661,76,3217900,20,0,30,784,1


In [4]:
data.groupby(["loan_status"]).size()

loan_status
0    34993
1    10000
dtype: int64

In [5]:
#Encoding Categorical Features using OnehotEncoder
categorical = data.iloc[:,[1,2,5,7,12]]
encode = OneHotEncoder(handle_unknown="ignore")
fit = encode.fit_transform(categorical)
encode_df = pd.DataFrame(fit.toarray(),columns=encode.get_feature_names_out(categorical.columns))

In [6]:
encode_df

,person_gender_female,person_gender_male,person_education_Associate,person_education_Bachelor,person_education_Doctorate,person_education_High School,person_education_Master,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_DEBTCONSOLIDATION,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,previous_loan_defaults_on_file_No,previous_loan_defaults_on_file_Yes
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44988,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
44989,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
44990,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
44991,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [7]:
data.columns

Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='str')

In [8]:
new_data = pd.concat([encode_df.astype("int"),data.drop(columns=["person_gender","person_education","person_home_ownership","previous_loan_defaults_on_file","loan_intent"])],axis=1)

In [9]:
new_data

,person_gender_female,person_gender_male,person_education_Associate,person_education_Bachelor,person_education_Doctorate,person_education_High School,person_education_Master,person_home_ownership_MORTGAGE,person_home_ownership_OTHER,person_home_ownership_OWN,...,previous_loan_defaults_on_file_Yes,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
0,1,0,0,0,0,0,1,0,0,0,...,0,22.0,71948.0,0,3217900.00,16.02,0.49,3.0,561,1
1,1,0,0,0,0,1,0,0,0,1,...,1,21.0,12282.0,0,91940.00,11.14,0.08,2.0,504,0
2,1,0,0,0,0,1,0,1,0,0,...,0,25.0,12438.0,3,505670.00,12.87,0.44,3.0,635,1
3,1,0,0,1,0,0,0,0,0,0,...,0,23.0,79753.0,0,3217900.00,15.23,0.44,2.0,675,1
4,0,1,0,0,0,0,1,0,0,0,...,0,24.0,66135.0,1,3217900.00,14.27,0.53,4.0,586,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44988,0,1,1,0,0,0,0,0,0,0,...,0,27.0,47971.0,6,1379100.00,15.66,0.31,3.0,645,1
44989,1,0,1,0,0,0,0,0,0,0,...,0,37.0,65800.0,17,827460.00,14.07,0.14,11.0,621,1
44990,0,1,1,0,0,0,0,0,0,0,...,0,33.0,56942.0,7,254765.74,10.02,0.05,10.0,668,1
44991,0,1,0,1,0,0,0,0,0,0,...,0,29.0,33164.0,4,1103280.00,13.23,0.36,6.0,604,1


In [10]:
#Scaling
Std = StandardScaler()
Std_ = Std.fit_transform(new_data.loc[:,["person_income","loan_amnt","credit_score"]])
df_std = pd.DataFrame(Std_,columns=Std.get_feature_names_out())
df_std

,person_income,loan_amnt,credit_score
0,-0.125715,4.025004,-1.420299
1,-1.067987,-1.359230,-2.551210
2,-1.065523,-0.646611,0.047901
3,-0.002455,4.025004,0.841522
4,-0.217516,4.025004,-0.924286
...,...,...,...
44988,-0.504370,0.857807,0.246306
44989,-0.222807,-0.092352,-0.229867
44990,-0.362696,-1.078775,0.702639
44991,-0.738209,0.382728,-0.567156


In [11]:
dt1 = pd.concat([new_data.drop(columns=["person_income","loan_amnt","credit_score","loan_status"]),df_std,new_data["loan_status"]],axis=1)

In [13]:
dt1.to_csv("Data/Transformation.csv",index=False)
new_data.to_csv("Data/Transformation_org.csv",index=False)

In [ ]:
np.zeros((1,27),dtype="float")

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

## Final Report: Data Transformation
<li>
    Encoded Categorical Features "Gender","Married","Dependents","Self_Employed","Property_Area","Education" using OneHotEncoder
</li>
<li>Imbalance values found in numerical features like "Credit_history" and other numerical Features.Standard Scaler is used for Scaling numerical features"           ApplicantIncome","LoanAmount","Loan_Amount_Term","CoapplicantIncome"</li>